In [70]:
import importlib
from utils import *
from groq import Groq
from prompts.valid_prompt import valid_reasoning_prompts
from prompts.invalid_prompt import invalid_reasoning_prompts
import re
import time

try:
    import config as app_config
except ModuleNotFoundError:
    import backend.config as app_config

app_config = importlib.reload(app_config)
groq = app_config.groq


In [71]:
#getting the groq client
if not groq:
    raise ValueError("GROQ_API_KEY is missing. Add it to backend/.env.local")

client = Groq(api_key=groq)

In [72]:
#function to generate valid reasoning
def generate_valid_reasoning(problem:str,solution:str):
    chat_completion=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"system",
                "content":valid_reasoning_prompts
            },
            {
                "role":"user",
                "content":f"Problem: {problem}\nSolution: {solution}"
            }
        ]
    )

    return chat_completion.choices[0].message.content

In [73]:
#function to generate invalid reasoning
def generate_invalid_reasoning(problem:str,solution:str):
    chat_completion=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"system",
                "content":invalid_reasoning_prompts
            },
            {
                "role":"user",
                "content":f"Problem: {problem}\nSolution: {solution}"
            }
        ]
    )

    return chat_completion.choices[0].message.content

In [74]:
#function to extract reasoning and answer from LLM response
def extract_reasoning_and_answer(content: str):
    if not content or not content.strip():
        return None, None

    # Pattern 1: Split on labeled answer — handles:
    #   Answer: | **Answer**: | **Answer:** | Final Answer: etc.
    parts = re.split(r'(?i)\*{0,2}\s*(?:final\s+)?answer\s*\*{0,2}\s*:\s*\*{0,2}', content)
    if len(parts) > 1:
        answer_part = parts[-1].strip()
        reasoning_part = "Answer:".join(parts[:-1]).strip()
        reasoning_part = re.sub(r'(?i)^\*{0,2}\s*reasoning\s*\*{0,2}\s*:\s*\*{0,2}\s*', '', reasoning_part)
        if reasoning_part and answer_part:
            return reasoning_part, answer_part

    # Pattern 2: \"The answer is X\" at the end
    m = re.search(r'(?i)the\s+answer\s+is\s*[:\s]*(.+)$', content)
    if m:
        answer_part = m.group(1).strip().rstrip('.')
        reasoning_part = content[:m.start()].strip()
        reasoning_part = re.sub(r'(?i)^\*{0,2}\s*reasoning\s*\*{0,2}\s*:\s*\*{0,2}\s*', '', reasoning_part)
        if reasoning_part and answer_part:
            return reasoning_part, answer_part

    # Pattern 3: Last line is the answer
    lines = [l.strip() for l in content.strip().split('\\n') if l.strip()]
    if len(lines) >= 2:
        last_line = lines[-1]
        last_line_clean = re.sub(
            r'(?i)^\*{0,2}\s*(?:answer|result|final answer)\s*\*{0,2}\s*:?\s*',
            '', last_line
        ).strip()
        if last_line_clean and len(last_line_clean) < 50:
            reasoning_part = '\\n'.join(lines[:-1])
            reasoning_part = re.sub(r'(?i)^\*{0,2}\s*reasoning\s*\*{0,2}\s*:\s*\*{0,2}\s*', '', reasoning_part)
            return reasoning_part, last_line_clean

    return content.strip(), None

In [75]:
#function to check if the generated answer matches the ground truth answer
import math

def clean_number(text):
    if not text:
        return None
    numbers = re.findall(r'-?\d+(?:\.\d+)?', str(text))
    if numbers:
        return float(numbers[-1])
    return None

def normalize_text(text):
    if not text:
        return ""
    return re.sub(r'[^\w]', '', str(text).lower().strip())

def is_match(generated_answer, ground_truth):
    gen_num = clean_number(generated_answer)
    truth_num = clean_number(ground_truth)

    # Numeric comparison with tolerance
    if gen_num is not None and truth_num is not None:
        return math.isclose(gen_num, truth_num, rel_tol=1e-6)

    # Fallback to text comparison
    return normalize_text(generated_answer) == normalize_text(ground_truth)

In [76]:
#loading the preprocessed data
dataset=load_data("../Data/processed/preprocessed_data.csv")

In [ ]:
#generating the synthetic reasoning data for Critic model pipeline

valid_reasonings = []
invalid_reasonings = []
valid_reasoning_results = []
invalid_reasoning_results = []
skipped_rows = []    
original_indices = []

MAX_RETRIES = 2

data = dataset[2:5]
for index, row in data.iterrows():
    problem = row["INSTRUCTION"]
    solution = str(row["RESPONSE"]).strip()
    success = False

    for attempt in range(MAX_RETRIES + 1):
        try:
            valid_reasoning = generate_valid_reasoning(problem, solution)
            invalid_reasoning = generate_invalid_reasoning(problem, solution)

            # Extract the reasoning and result from the generated reasoning
            valid_reasoning_steps, valid_reasoning_result = extract_reasoning_and_answer(valid_reasoning)
            invalid_reasoning_steps, invalid_reasoning_result = extract_reasoning_and_answer(invalid_reasoning)

            if not valid_reasoning_steps or not invalid_reasoning_steps:
                if attempt < MAX_RETRIES:
                    print(f"Row {index}: Extraction failed (attempt {attempt+1}), retrying...")
                    time.sleep(2)
                    continue
                print(f"Row {index}: Extraction failed after {MAX_RETRIES+1} attempts.")
                skipped_rows.append({"index": index, "reason": "extraction_failed", "problem": problem[:80]})
                break

            valid_match = is_match(valid_reasoning_result, solution)
            invalid_match = is_match(invalid_reasoning_result, solution)

            if not valid_match:
                if attempt < MAX_RETRIES:
                    print(f"Row {index}: Valid answer mismatch (attempt {attempt+1}), retrying...")
                    time.sleep(2)
                    continue
                print(f"Row {index}: Valid answer mismatch after all attempts. Got '{valid_reasoning_result}', expected '{solution}'")
                skipped_rows.append({"index": index, "reason": "valid_mismatch", "got": str(valid_reasoning_result), "expected": solution})
                break

            if not invalid_match:
                if attempt < MAX_RETRIES:
                    print(f"Row {index}: Invalid answer mismatch (attempt {attempt+1}), retrying...")
                    time.sleep(2)
                    continue
                print(f"Row {index}: Invalid answer mismatch after all attempts. Got '{invalid_reasoning_result}', expected '{solution}'")
                skipped_rows.append({"index": index, "reason": "invalid_mismatch", "got": str(invalid_reasoning_result), "expected": solution})
                break

            # All checks passed
            valid_reasonings.append(valid_reasoning_steps)
            invalid_reasonings.append(invalid_reasoning_steps)
            valid_reasoning_results.append(valid_reasoning_result)
            invalid_reasoning_results.append(invalid_reasoning_result)
            original_indices.append(index)
            success = True
            break

        except Exception as e:
            if attempt < MAX_RETRIES:
                print(f"Row {index}: API error (attempt {attempt+1}): {e}. Retrying...")
                time.sleep(5)
            else:
                print(f"Row {index}: API error after all attempts: {e}")
                skipped_rows.append({"index": index, "reason": "api_error", "error": str(e)})

    time.sleep(1)  # Rate limit between rows

print(f"\nProcessed: {len(valid_reasonings)} rows | Skipped: {len(skipped_rows)} rows | Total: {len(valid_reasonings) + len(skipped_rows)} rows")
if skipped_rows:
    print("Skipped row details:")
    for s in skipped_rows:
        print(f"  Row {s['index']}: {s['reason']}")
else:
    print("All rows processed successfully!")

if len(valid_reasonings) > 0:
    print("\n--- Sample Output ---")
    print("Valid Reasoning Steps:", valid_reasonings[0][:200], "...")
    print("Valid Reasoning Result:", valid_reasoning_results[0])
    print("Invalid Reasoning Steps:", invalid_reasonings[0][:200], "...")
    print("Invalid Reasoning Result:", invalid_reasoning_results[0])
